In [105]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [106]:
load_dotenv(override=True)

openai = OpenAI(api_key=os.getenv("OPEN_API_KEY"))

In [107]:
reader = PdfReader("me/jaideep-resume-2.pdf")
aboutme1 = ""
for page in reader.pages:
    text=page.extract_text()
    if text:
        aboutme1 += text 


aboutme1



'Jaideep Tripurani\nChicago, IL♂phone(346)-236-5831/envel⌢pejaideeptripurani10@gmail.com/linkedinLinkedIn/githubGitHub\nProfessional Summary\nFull Stack Engineerwith1+ yearof experience buildingagentic AI systems,RAG pipelines, andLLM-powered workflows\nbacked by a strongJava Spring BootandPythonbackend foundation. At Walmart, integratedcontext-aware RAG agentswith\nquery decomposition,multi-step retrieval, andFAISS vector searchthat automated25%of recurring operational workflows.\nEngineeredKafkaevent-driven microservices processing over1M events/daywith99.9% uptime. Built a production\nmulti-domain triage agentusingGPT-4o,dynamic task routing, andrules-based escalation— shipped and demoed as part of\na national AI hackathon (May 2026) and secured below100 rank out of 3000. Excited to build agents that do real work for real users\nat an AI-first startup.\nWork Experience\nFull Stack Engineer — Walmart Oct 2025 – Present\nContract Vendor: Iconsoft Inc\n–Architected and deployedcontext-

In [108]:
reader = PdfReader("me/jaideep-resume1.pdf")


for page in reader.pages:
    text=page.extract_text()
    if text:
        aboutme1 += text 


aboutme1

'Jaideep Tripurani\nChicago, IL♂phone(346)-236-5831/envel⌢pejaideeptripurani10@gmail.com/linkedinLinkedIn/githubGitHub\nProfessional Summary\nFull Stack Engineerwith1+ yearof experience buildingagentic AI systems,RAG pipelines, andLLM-powered workflows\nbacked by a strongJava Spring BootandPythonbackend foundation. At Walmart, integratedcontext-aware RAG agentswith\nquery decomposition,multi-step retrieval, andFAISS vector searchthat automated25%of recurring operational workflows.\nEngineeredKafkaevent-driven microservices processing over1M events/daywith99.9% uptime. Built a production\nmulti-domain triage agentusingGPT-4o,dynamic task routing, andrules-based escalation— shipped and demoed as part of\na national AI hackathon (May 2026) and secured below100 rank out of 3000. Excited to build agents that do real work for real users\nat an AI-first startup.\nWork Experience\nFull Stack Engineer — Walmart Oct 2025 – Present\nContract Vendor: Iconsoft Inc\n–Architected and deployedcontext-

In [109]:
print(aboutme1)

Jaideep Tripurani
Chicago, IL♂phone(346)-236-5831/envel⌢pejaideeptripurani10@gmail.com/linkedinLinkedIn/githubGitHub
Professional Summary
Full Stack Engineerwith1+ yearof experience buildingagentic AI systems,RAG pipelines, andLLM-powered workflows
backed by a strongJava Spring BootandPythonbackend foundation. At Walmart, integratedcontext-aware RAG agentswith
query decomposition,multi-step retrieval, andFAISS vector searchthat automated25%of recurring operational workflows.
EngineeredKafkaevent-driven microservices processing over1M events/daywith99.9% uptime. Built a production
multi-domain triage agentusingGPT-4o,dynamic task routing, andrules-based escalation— shipped and demoed as part of
a national AI hackathon (May 2026) and secured below100 rank out of 3000. Excited to build agents that do real work for real users
at an AI-first startup.
Work Experience
Full Stack Engineer — Walmart Oct 2025 – Present
Contract Vendor: Iconsoft Inc
–Architected and deployedcontext-aware RAG pipe

In [110]:
with open("me/aboutme.txt","r", encoding="utf-8") as f:
    summary = f.read()


In [111]:
name = "Jaideep Tripurani"

In [112]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## resume:\n{aboutme1}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [113]:
def chat(message,history):
    messages = [{"role":"system","content":system_prompt}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model="gpt-4o-mini",messages=messages)
    return response.choices[0].message.content


In [114]:
gr.ChatInterface(chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [116]:
from pydantic import BaseModel
class Evaluation(BaseModel):
    is_accepatble:bool
    feedback:str


In [117]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{aboutme1}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [118]:
def evaluator(message,history,reply):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt


In [119]:
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [122]:
def evaluate(reply,message,history)-> Evaluation:
    messages = [{"role":"system","content":evaluator_system_prompt}] + [{"role":"user","content":evaluator(message,history,reply)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed


In [127]:
def rerun(message,history,reply,feedback):
    updated_system_prompt += system_prompt + "\nyour previour reply was rejected by the evauator,please check and reply back again\n\n"
    updated_system_prompt += f"the feedback you got was {feedback}\n\n"
    updated_system_prompt += f"the previours reply you sent was {reply}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content





In [129]:
def chat(message,history):
    system = system_prompt
    messages = [{"role":"system","content":system}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model="gpt-4o", messages=messages)
    reply = response.choices[0].message.content

    eval = evaluate(reply,message,history)

    if eval.is_accepatble:
        print("passed eval")
    else:
        print("eval failed")
        reply = rerun(message,history,reply,eval.feedback)
    return reply








In [ ]:
gr.ChatInterface(chat,type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


passed eval
passed eval
